# Part B demo: 3D scene and 28 GHz ray tracing (Team 15)
Run the cells in order with **Shift + Enter**. Drag in a 3D view to rotate, scroll to zoom, right-drag to pan.

## 1. Build the street from code
Writes `scene/scene.xml` (materials) and `scene/meshes/*.ply` (buildings, kiosk, truck, ground).

In [ ]:
%run build_scene.py

## 2. Load the scene in Sionna RT and show it in 3D

In [ ]:
import sionna.rt as rt, mitsuba as mi
from config import FC, BS, TRACE

scene = rt.load_scene("scene/scene.xml", merge_shapes=False)
for name, obj in scene.objects.items():
    print(f"{name:<10} {obj.radio_material.name}")
scene.preview(resolution=(1000, 700))

## 3. Trace 28 GHz rays from the BS to four users and show them in 3D
`clip_at=12` cuts the buildings at 12 m so the rays inside the side street are visible. Remove it to see full buildings.

In [ ]:
scene = rt.load_scene("scene/scene.xml", merge_shapes=False)   # fresh scene every run
scene.frequency = FC
scene.tx_array = rt.PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = rt.PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

scene.add(rt.Transmitter("BS", position=mi.Point3f(*map(float, BS)), display_radius=2.5))
users = {"U1": (40, 15, 1.5), "U2": (40, 40, 1.5), "U3": (40, 62, 1.5), "U4": (40, 90, 1.5)}
for name, p in users.items():
    scene.add(rt.Receiver(name, position=mi.Point3f(*map(float, p)), display_radius=1.5))

paths = rt.PathSolver()(scene, **TRACE)
scene.preview(paths=paths, clip_at=12, resolution=(1000, 700))

## 4. The numbers: SNR at each user
Also saves `out/demo_rays.png` and `out/scene_3d.html`.

In [ ]:
%run demo_live.py